# Curvilinear Boundary Conditions

Connect ghost zones, parity, and radiation stencils to curvilinear generated-code boundary conditions.

Navigation: [Index](../index.ipynb) | Previous: [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb) | Next: [GeneralRFM and Fisheye Coordinates](generalrfm_and_fisheye.ipynb)


## Parity and Radiation Stencils
Parity classes tell an inner ghost-zone fill whether a tensor component keeps or flips sign. Outer radiation conditions use one-sided finite-difference coefficients.

## Import Boundary-Condition Registration Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [ ]:
import nrpy.c_function as cfc
import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    apply_bcs_outerradiation_and_inner as outer_bcs,
)
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    bcstruct_set_up,
    register_all,
)


## Step 2: List parity classes

Parity classes tell boundary routines how tensor components change sign across coordinate faces.

In [ ]:
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fp_type", "double")
parity_classes = [
    "scalar",
    "x0 vector or one-form component",
    "x1 vector or one-form component",
    "x2 vector or one-form component",
    "x0x0 symmetric rank-2 component",
    "x0x1 symmetric rank-2 component",
    "x0x2 symmetric rank-2 component",
    "x1x1 symmetric rank-2 component",
    "x1x2 symmetric rank-2 component",
    "x2x2 symmetric rank-2 component",
]
print("parity classes:")
for index, name in enumerate(parity_classes):
    print(index, name)


## Step 3: Inspect spherical parity assignments

The generated assignment block maps each parity class to symbolic dot products.

In [ ]:
parity_code = bcstruct_set_up.parity_conditions_symbolic_dot_products("Spherical")
assignment_block = parity_code[parity_code.index("{") :]
print("complete Spherical parity assignment block:")
print(assignment_block)


## Step 4: Compute an offset finite-difference stencil

The coefficients show how an off-center derivative is represented near a boundary.

In [ ]:
coeffs, offsets = outer_bcs.get_arb_offset_FD_coeffs_indices(4, -1, 1)
print("fourth-order offset derivative coefficients:")
for coeff, offset in zip(coeffs, offsets):
    print(coeff, offset)


## Step 5: Register curvilinear boundary functions

The printed names are the generated helpers available to apply boundary conditions.

In [ ]:
for name in list(cfc.CFunction_dict):
    if (
        "bc" in name.lower()
        or "boundary" in name.lower()
        or "inbounds" in name.lower()
        or "parity" in name.lower()
    ):
        cfc.CFunction_dict.pop(name, None)
register_all.register_C_functions({"Spherical"}, radiation_BC_fd_order=4)
print("registered boundary functions:")
for name in sorted(
    key
    for key in cfc.CFunction_dict
    if "bc" in key.lower()
    or "boundary" in key.lower()
    or "inbounds" in key.lower()
    or "parity" in key.lower()
):
    print(name)
    print(cfc.CFunction_dict[name].function_prototype.splitlines()[0])


The parity classes describe inner ghost-zone signs, and the registered boundary functions show the generated calls that apply those rules and outer radiation conditions in a curvilinear project.


## Continue to Fisheye Coordinates
- [Boundary Conditions and Convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
- [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb)
